# RMT-PPAD migration NB79 - Phase P0 (baseline reproduction)

**Purpose.** Reproduce the published RMT-PPAD numbers on BDD100K so we have a
fixed reference target. The full migration plan (P0-P8) is in
`yolop_vehicle_lane/stage2/rmt_ppad_migration/README.md`.

**Acceptance criterion (from the source plan):** the printed numbers should be
approximately:
- Detection: Recall ~= 0.954, mAP50 ~= 0.849
- Drivable area mIoU ~= 0.926 (this will be removed in P4+)
- Lane line IoU ~= 0.568, ACC ~= 0.847

**Output:** `yolop_vehicle_lane/stage2/rmt_ppad_migration/results/baseline_metrics.json`

**Time:** ~30-90 min wall clock (mostly downloads + 5000-image val pass).

### Cell 1: Mount Drive, install RMT-PPAD deps, set REPO_ROOT
RMT-PPAD's environment.yml requires Python 3.8 + PyTorch 2.4. Colab usually
ships PyTorch 2.x already, so we only need to pip-install their extras.

In [1]:
import os, sys, subprocess
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Two pip installs because mmcv often pulls in build deps that conflict
# with the lighter list if combined.
#
# 1) Ultralytics-style runtime deps (RMT-PPAD reuses these).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pyyaml', 'tqdm', 'matplotlib', 'opencv-python-headless', 'scipy',
    'pandas', 'seaborn', 'requests', 'thop', 'psutil', 'py-cpuinfo'])

# 2) mmcv -- RMT-PPAD's MTDETR (and CLRKDNet's CLRHead) imports
#    mmcv.cnn.ConvModule. Without this, the first `from ultralytics import
#    MTDETR` crashes at module load with ModuleNotFoundError, which is what
#    happened in NB79 v1 (subprocess exited rc=1 in 24 s during cell 5).
try:
    import mmcv  # noqa: F401
    print(f'[ok] mmcv already installed: {mmcv.__version__}')
except ImportError:
    print('[install] mmcv (~1-3 min wheel build)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
    import mmcv  # noqa: F401
    print(f'[ok] mmcv installed: {mmcv.__version__}')

print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
[install] mmcv (~1-3 min wheel build)...
[ok] mmcv installed: 2.2.0
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


### Cell 2: Vendor RMT-PPAD source (shallow copy)
Copies `external_repos/RMT-PPAD-main/ultralytics/` to a writable working folder
under `stage2/rmt_ppad_migration/vendor/RMT-PPAD/`. P2-P8 will modify files here;
the original `external_repos/` copy stays untouched.

In [2]:
import sys, os

RMT_PPAD_SRC = '/content/drive/MyDrive/EcoCAR/external_repos/RMT-PPAD-main'
RMT_PPAD_DST = ('/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/'
                'stage2/rmt_ppad_migration/vendor/RMT-PPAD')

if not os.path.isdir(RMT_PPAD_SRC):
    raise FileNotFoundError(f'{RMT_PPAD_SRC} not present. Did you clone the repo into Drive?')

cmd = [sys.executable, '-u',
       'stage2/rmt_ppad_migration/P0_baseline/vendor_rmt_ppad.py',
       '--src', RMT_PPAD_SRC, '--dst', RMT_PPAD_DST]
log = os.path.join(LOG_DIR, 'NB79_vendor.log')
run_streaming(cmd, log_path=log)

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P0_baseline/vendor_rmt_ppad.py --src /content/drive/MyDrive/EcoCAR/external_repos/RMT-PPAD-main --dst /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB79_vendor.log
[file] README.md (9.3 KB)
[file] environment.yml (6.7 KB)
[file] pyproject.toml (7.6 KB)
[file] LICENSE (33.7 KB)
[file] CITATION.cff (0.7 KB)
[file] CONTRIBUTING.md (10.1 KB)
[run_streaming] still running; no child output yet. This usually means the first dataloader/model step is still working.
[dir]  ultralytics: 251 files, 2.5 MB

[done] 251 files copied / updated, 2.5 MB
       vendored at: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD
[run_streaming] return_code=0


0

### Cell 3: Download RMT-PPAD pretrained weights + BDD detection labels + masks
Three SharePoint links from RMT-PPAD's README. The downloader appends
`?download=1` to force binary download instead of HTML preview.

If any of the three fails automatically, the script prints a manual-fallback URL
you can open in a browser; save the file to `DOWNLOADS_DIR` and re-run this cell.

In [3]:
import sys, os

DOWNLOADS_DIR = '/content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights'
os.makedirs(DOWNLOADS_DIR, exist_ok=True)

cmd = [sys.executable, '-u',
       'stage2/rmt_ppad_migration/P0_baseline/download_rmt_ppad_pretrained.py',
       '--dest', DOWNLOADS_DIR]
log = os.path.join(LOG_DIR, 'NB79_download.log')
run_streaming(cmd, log_path=log)

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P0_baseline/download_rmt_ppad_pretrained.py --dest /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB79_download.log
[plan] downloading 3 target(s) into /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights
[skip] /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/rmt_ppad_best.pt already present (71.0 MB)
[skip] /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip already present (44.1 MB)
[skip] /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_seg_masks.zip already present (592.7 MB)

=== Summary ===
  OK  /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/rmt_ppad_best.pt
  OK  /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip
  OK  /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_seg_masks.zip
[run_streaming] return_code=0

0

### Cell 4: Extract RMT-PPAD's labels/masks into LOCAL Colab filesystem

**Critical**: writing tens of thousands of small files into Drive is brutally slow
(every file is a separate Drive API call). Both extractions go to `/content/` (the
Colab instance's local SSD) which is ~20-50x faster.

RMT-PPAD's `BDD_full.yaml` expects this layout at `BDD_LOCAL`:
- `BDD_seg_mask/images/{train2017,val2017}/*.jpg`  - from your existing BDD image source
- `BDD_seg_mask/labels/{train2017,val2017}/*.txt`  - from `BDD_detection_labels.zip`
- `BDD_seg_mask/mask/lane/{train2017,val2017}/*.png`  - from `BDD_seg_masks.zip`
- `BDD_seg_mask/mask/drivable/{train2017,val2017}/*.png`  - from `BDD_seg_masks.zip`

Set `IMAGES_SOURCE` to point at your existing BDD raw images (directory or tarball).
The cell tries (in order): symlink an existing dir; extract a tarball locally;
raise a clear error. Idempotent: re-runs skip what already exists.

In [4]:
import os, zipfile, subprocess, time, shutil
from pathlib import Path

DOWNLOADS_DIR = Path('/content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights')
IMAGES_SOURCE = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
BDD_LOCAL = Path('/content/BDD_seg_mask')
BDD_LOCAL.mkdir(parents=True, exist_ok=True)

# CRITICAL: scratch dir name MUST NOT contain the substring 'images'.
# RMT-PPAD's dataset.py line 187 does `im_file.replace('images', task_name)`
# which globally replaces every 'images' occurrence; if the image file
# resolves through a path containing 'images' twice, the second one gets
# mangled too. We use 'bdd_extract_scratch' (no 'images' substring).
SCRATCH = Path('/content/bdd_extract_scratch')

def _count(p, pattern, cap=5000):
    n = 0
    for _ in p.rglob(pattern):
        n += 1
        if n >= cap:
            break
    return n

# Step 1: extract labels and masks zips into LOCAL /content/.
for zip_name, marker_sub, file_ext in [
    ('BDD_detection_labels.zip', 'labels', 'txt'),
    ('BDD_seg_masks.zip',        'mask',   'png'),
]:
    zp = DOWNLOADS_DIR / zip_name
    marker_dir = BDD_LOCAL / marker_sub
    if not zp.exists():
        print(f'[skip] {zip_name} not present in {DOWNLOADS_DIR}; re-run cell 3 first.')
        continue
    if marker_dir.exists() and _count(marker_dir, f'*.{file_ext}') >= 10:
        print(f'[skip-extracted] {zip_name} -> {marker_dir} already populated')
        continue
    t0 = time.time()
    print(f'[extract-local] {zp}  ->  {BDD_LOCAL}  (zip)')
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(BDD_LOCAL)
    print(f'  done in {time.time()-t0:.1f}s; {_count(marker_dir, f"*.{file_ext}")} {file_ext} files now present')

# Step 2: BDD images. Detect existing or extract/relocate.
img_train = BDD_LOCAL / 'images' / 'train2017'
img_val   = BDD_LOCAL / 'images' / 'val2017'

def _is_safe_jpg_dir(d):
    'A jpg dir is safe iff its absolute path has exactly one `images` segment.'
    if not d.exists():
        return False
    abs_path = str(d.resolve())
    return abs_path.count('/images/') == 1 and _count(d, '*.jpg') > 0

# If train/val images are already set up safely, do nothing.
if _is_safe_jpg_dir(img_train) and _is_safe_jpg_dir(img_val):
    print(f'[ok] images already at {img_train.parent} ({_count(img_train, "*.jpg")}+ train jpgs)')
elif IMAGES_SOURCE is None:
    print(f'[warn] IMAGES_SOURCE = None; populate {img_train} and {img_val} yourself before cell 5.')
else:
    src = Path(IMAGES_SOURCE)
    if not src.exists():
        raise FileNotFoundError(
            f'IMAGES_SOURCE = {src} does not exist. Set it to your BDD image dir/tarball, '
            'or download BDD100K from bdd-data.berkeley.edu.')

    # Remove any stale symlinks from NB79 v1 (these pointed at
    # /content/bdd_images_scratch which had the path-replace bug).
    for stale in (img_train, img_val):
        if stale.is_symlink():
            print(f'[unlink] removing stale symlink {stale} -> {os.readlink(stale)}')
            stale.unlink()

    if src.is_file() and src.suffix in ('.tar', '.gz', '.tgz'):
        if SCRATCH.exists() and _count(SCRATCH, '*.jpg') > 100:
            print(f'[skip-scratch] {SCRATCH} already has images')
        else:
            SCRATCH.mkdir(parents=True, exist_ok=True)
            t0 = time.time()
            print(f'[extract-local] {src}  ->  {SCRATCH}  (tar)')
            subprocess.check_call(['tar', '-xf', str(src), '-C', str(SCRATCH)])
            print(f'  done in {time.time()-t0:.1f}s')

        # Discover the actual jpg directories the tar created.
        def _find_split_dir(root, candidates):
            for name in candidates:
                for h in root.rglob(name):
                    if h.is_dir() and _count(h, '*.jpg') > 0:
                        return h
            return None
        train_src = _find_split_dir(SCRATCH, ('train2017', 'train', 'training'))
        val_src   = _find_split_dir(SCRATCH, ('val2017', 'val', 'validation'))
        if train_src is None or val_src is None:
            raise RuntimeError(
                f'Could not auto-locate train/val image dirs under {SCRATCH}. '
                f'List the structure and either rename or set IMAGES_SOURCE to '
                f'a directory laid out as <root>/{{train2017,val2017}}/*.jpg.')

        # HARDLINK the jpgs into BDD_LOCAL/images/<split>/. Hardlinks share
        # the same inode so no disk doubled, and the path RMT-PPAD's dataset.py
        # sees is `/content/BDD_seg_mask/images/<split>/X.jpg` -- exactly one
        # 'images' segment, so .replace('images','mask/drivable') works.
        for split_src, split_dst in ((train_src, img_train), (val_src, img_val)):
            split_dst.parent.mkdir(parents=True, exist_ok=True)
            split_dst.mkdir(parents=True, exist_ok=True)
            existing = _count(split_dst, '*.jpg')
            source_count = _count(split_src, '*.jpg')
            if existing >= source_count * 0.95:
                print(f'[skip-linked] {split_dst}: {existing} jpgs already present (>=95% of {source_count})')
                continue
            print(f'[hardlink] {split_src} -> {split_dst}  ({source_count} files)')
            t0 = time.time()
            for jpg in split_src.glob('*.jpg'):
                target = split_dst / jpg.name
                if target.exists():
                    continue
                try:
                    os.link(jpg, target)  # hardlink: same inode, no double disk
                except OSError:
                    # cross-device or other FS quirk -> fall back to copy
                    shutil.copy2(jpg, target)
            print(f'  done in {time.time()-t0:.1f}s; now {_count(split_dst, "*.jpg")} jpgs at {split_dst}')

    elif src.is_dir():
        # User-supplied directory: copy/link directly to BDD_LOCAL.
        for sub_name, sub_dst in (('train2017', img_train), ('val2017', img_val)):
            sub_src = src / sub_name
            if not sub_src.exists():
                for alt in ('train', 'training'):
                    if (src / alt).exists():
                        sub_src = src / alt
                        break
            if not sub_src.exists():
                print(f'[miss] {sub_src} not found; cannot wire {sub_dst}')
                continue
            sub_dst.parent.mkdir(parents=True, exist_ok=True)
            sub_dst.mkdir(parents=True, exist_ok=True)
            for jpg in sub_src.glob('*.jpg'):
                target = sub_dst / jpg.name
                if target.exists():
                    continue
                try:
                    os.link(jpg, target)
                except OSError:
                    shutil.copy2(jpg, target)
            print(f'[hardlink/copy] {sub_dst}: {_count(sub_dst, "*.jpg")} jpgs')
    else:
        raise RuntimeError(f'Unsupported IMAGES_SOURCE type: {src}')

# Final inspection + safety check that no path contains a duplicate 'images'.
print(f'\n[final-inspect] {BDD_LOCAL}')
EXPECTED = [
    ('images/train2017',          'jpg', 100),
    ('images/val2017',            'jpg',  10),
    ('labels/train2017',          'txt', 100),
    ('labels/val2017',            'txt',  10),
    ('mask/lane/train2017',       'png', 100),
    ('mask/lane/val2017',         'png',  10),
    ('mask/drivable/train2017',   'png', 100),
    ('mask/drivable/val2017',     'png',  10),
]
fail = 0
for sub, ext, min_count in EXPECTED:
    pp = BDD_LOCAL / sub
    if not pp.exists():
        print(f'  MISSING  {sub}')
        fail += 1
        continue
    n = _count(pp, f'*.{ext}')
    status = 'OK     ' if n >= min_count else 'PARTIAL'
    print(f'  {status}  {sub:30s} ({n} {ext} files; need >={min_count})')
    if status.strip() != 'OK':
        fail += 1

# Safety: confirm a sample image's RESOLVED absolute path contains 'images' exactly once.
sample = next(img_val.glob('*.jpg'), None)
if sample is not None:
    resolved = str(sample.resolve())
    n_images_segs = resolved.count('/images/')
    if n_images_segs == 1:
        print(f'\n[path-safety] OK  sample {sample.name} resolves to {resolved}')
        print(f'              `/images/` appears exactly once -> str.replace will be clean.')
    else:
        print(f'\n[path-safety] FAIL sample resolves to {resolved}')
        print(f'              `/images/` appears {n_images_segs} times -> RMT-PPAD will mangle paths.')
        print(f'              Move or rename the scratch dir so it does not contain `images`.')
        fail += 1

if fail:
    print(f'\n[warn] {fail} check(s) failed. Cell 5 may fail.')
else:
    print(f'\n[ok] BDD layout complete at {BDD_LOCAL}')

[extract-local] /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_detection_labels.zip  ->  /content/BDD_seg_mask  (zip)
  done in 6.0s; 5000 txt files now present
[extract-local] /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/BDD_seg_masks.zip  ->  /content/BDD_seg_mask  (zip)
  done in 16.7s; 5000 png files now present
[extract-local] /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar  ->  /content/bdd_extract_scratch  (tar)
  done in 68.7s
[hardlink] /content/bdd_extract_scratch/images/train -> /content/BDD_seg_mask/images/train2017  (5000 files)
  done in 1.0s; now 5000 jpgs at /content/BDD_seg_mask/images/train2017
[hardlink] /content/bdd_extract_scratch/images/val -> /content/BDD_seg_mask/images/val2017  (5000 files)
  done in 0.1s; now 5000 jpgs at /content/BDD_seg_mask/images/val2017

[final-inspect] /content/BDD_seg_mask
  OK       images/train2017               (5000 jpg files; need >=100)
  OK       images/val2017                 (5000 jpg fi

### Cell 5: Run RMT-PPAD's MTDETR.val() on the BDD val split
Patches `BDD_full.yaml` to point at our actual BDD root, then invokes
MTDETR.val(...) via the runner script. Wall-clock ~10-30 minutes on a single
GPU with the full 5000-image val set.

In [5]:
import os, sys
from pathlib import Path

# BDD_ROOT is LOCAL (populated in cell 4). Drive paths here would make every image
# read a Drive API call -- killing val throughput.
BDD_ROOT = '/content/BDD_seg_mask'
RMT_PPAD_DST = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD'
CHECKPOINT = '/content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/rmt_ppad_best.pt'
OUTPUT_JSON = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/results/baseline_metrics.json'

for path_, why in [
    (BDD_ROOT,     'cell 4 must have populated this local dir'),
    (RMT_PPAD_DST, 'cell 2 must have vendored the source'),
    (CHECKPOINT,   'cell 3 must have downloaded the .pt'),
]:
    if not Path(path_).exists():
        raise FileNotFoundError(f'prereq missing: {path_}  ({why})')

cmd = [sys.executable, '-u',
       'stage2/rmt_ppad_migration/P0_baseline/run_baseline_val.py',
       '--rmt-ppad-root', RMT_PPAD_DST,
       '--checkpoint', CHECKPOINT,
       '--bdd-root', BDD_ROOT,
       '--output-json', OUTPUT_JSON,
       '--batch', '1',
       '--imgsz', '640',
       '--mask-thr', '0.45,0.9']
log = os.path.join(LOG_DIR, 'NB79_baseline_val.log')
run_streaming(cmd, log_path=log)

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P0_baseline/run_baseline_val.py --rmt-ppad-root /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD --checkpoint /content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/rmt_ppad_best.pt --bdd-root /content/BDD_seg_mask --output-json /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/results/baseline_metrics.json --batch 1 --imgsz 640 --mask-thr 0.45,0.9
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB79_baseline_val.log
[yaml] wrote /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/datasets/BDD_full_local.yaml with path=/content/BDD_seg_mask
[runner] launching MTDETR.val subprocess (this can take 10-30 min on the full BDD val split, ~5000 images)...
[run_streaming] still running; no child output yet. This usually means the first dataloader/model step is still 

0

### Cell 6: Inspect parsed metrics
If the runner parsed numbers from stdout they'll be in `baseline_metrics.json`.
If parsing failed, the runner still wrote the full log; eyeball it manually.

In [6]:
import json
from pathlib import Path

OUTPUT_JSON = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/results/baseline_metrics.json'
if not Path(OUTPUT_JSON).exists():
    raise FileNotFoundError('baseline_metrics.json not produced; Cell 5 must complete first.')

rec = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
print(json.dumps(rec, indent=2))

print('\n=== Reference numbers from RMT-PPAD paper ===')
print('  Detection Recall = 0.954, mAP50 = 0.849')
print('  Drivable mIoU    = 0.926')
print('  Lane IoU         = 0.568')
print('  Lane ACC         = 0.847')

# P0 acceptance: numbers within +/- 2 absolute points of reference.
metrics = rec.get('metrics', {}) or {}
checks = {
    'detection_map50': (0.849, 0.02),
    'drivable_miou':   (0.926, 0.02),
    'lane_iou':        (0.568, 0.02),
    'lane_acc':        (0.847, 0.02),
}
print('\n[acceptance test]')
fails = 0
for key, (ref, tol) in checks.items():
    val = metrics.get(key)
    if val is None:
        print(f'  ??  {key}: not parsed (review log)')
        fails += 1
    elif abs(val - ref) <= tol:
        print(f'  OK  {key}: {val:.3f}  (reference {ref:.3f})')
    else:
        print(f'  X   {key}: {val:.3f}  (reference {ref:.3f}, tol {tol})')
        fails += 1
print('\n[P0 result]', 'PASS' if fails == 0 else f'FAIL ({fails} checks not OK)')

{
  "returncode": 0,
  "rmt_ppad_root": "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD",
  "checkpoint": "/content/drive/MyDrive/EcoCAR/downloads/rmt_ppad_weights/rmt_ppad_best.pt",
  "bdd_root": "/content/BDD_seg_mask",
  "imgsz": 640,
  "batch": 1,
  "mask_threshold": [
    0.45,
    0.9
  ],
  "metrics": {
    "detection_precision": 0.0345,
    "detection_recall": 0.954,
    "detection_map50": 0.849,
    "detection_map": 0.519
  },
  "log_path": "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/results/baseline_metrics.log"
}

=== Reference numbers from RMT-PPAD paper ===
  Detection Recall = 0.954, mAP50 = 0.849
  Drivable mIoU    = 0.926
  Lane IoU         = 0.568
  Lane ACC         = 0.847

[acceptance test]
  OK  detection_map50: 0.849  (reference 0.849)
  ??  drivable_miou: not parsed (review log)
  ??  lane_iou: not parsed (review log)
  ??  lane_acc: not parsed (review log)

[P0 result] FAIL (3 checks not 